# Transaction anomaly detection — notebook walkthrough

**Important:** This codebase includes a synthetic transaction generator, but **no real transaction dataset has been sourced or validated**. Under the project's real-data-only publication standard, this notebook will **not** run the generator or report fabricated results.

It instead explains and inspects the existing temporal pipeline. Run a real-data experiment only after acquiring a rights-cleared, appropriately labeled transaction file with all documented fields and timestamps. This notebook was added after the modular code, not evidence that notebook-first work occurred historically.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "README.md").exists() and (ROOT.parent / "README.md").exists():
    ROOT = ROOT.parent
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run the notebook from the project root or notebooks/ directory")
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

## 1. Real dataset requirements
Read the exact input schema and provenance requirements before preparing a CSV. **Do not rename a different dataset to fit the filename**.

In [ ]:
print((ROOT/"docs"/"DATA.md").read_text(encoding="utf-8")[:8000])

## 2. Optional exploratory data analysis on an ACTUAL, user-provided dataset
This cell fails safely if the dataset is missing; there is no synthetic fallback. The CSV must meet the contract in `docs/DATA.md`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
path = ROOT / "data" / "real" / "transactions.csv"
if not path.exists():
    raise FileNotFoundError(f"Real labeled dataset not provided: {path}. Skip experiment until verified.")
df = pd.read_csv(path)
expected={"transaction_id","timestamp","amount","international","card_present","account_age_days","is_anomaly"}
assert expected.issubset(df.columns), f"Missing columns: {sorted(expected-set(df.columns))}"
display(df.head())
display(df.isna().mean().sort_values(ascending=False))
display(df["is_anomaly"].value_counts(normalize=True).rename("class_fraction"))
df["timestamp"]=pd.to_datetime(df["timestamp"],utc=True,errors="raise")
df.sort_values("timestamp")["amount"].plot(figsize=(10,3),title="Transaction amount over time")
plt.show()

## 3. Temporal train, validation, and test
Use time cutoffs that correspond to the **actual real dataset**, with strictly forward validation/test. Inspect the modular code below; do not invent cutoffs or result claims.

In [ ]:
from transaction_anomaly import data, evaluation, pipeline
import inspect
print("Core modules:")
for module in (data,evaluation,pipeline):
    funcs=[n for n,v in vars(module).items() if inspect.isfunction(v) and getattr(v,"__module__","")==module.__name__]
    print(module.__name__,funcs[:25])

## 4. Running a real-data benchmark
After confirming the real dataset's provenance and date range, use `python -m transaction_anomaly.cli train --help` to select real cutoffs, then run from a terminal with `PYTHONPATH=src`. The pipeline compares prevalence and logistic baselines and reports average precision, recall, F1, calibration, and drift. This notebook does not claim that run occurred.

In [ ]:
import subprocess,os
env=os.environ.copy();env["PYTHONPATH"]=str(ROOT/"src")
r=subprocess.run([sys.executable,"-m","transaction_anomaly.cli","train","--help"],cwd=ROOT,env=env,text=True,capture_output=True)
print(r.stdout[:7000] or r.stderr[:3000])

## 5. Software mapping and interpretation
- `data.py`: schema validation, explicit feature choices, temporal partitions; synthetic generator exists but is **not acceptable substitute for real-data conclusions**.
- `evaluation.py`: metrics, PR curves, calibration and drift checks.
- `pipeline.py`: model selection, fitted artifacts, manifests, errors.
- `cli.py`: repeatable command-line interface.
- `tests/`: unit tests; not evidence of accuracy on real bank transactions.

**Status:** awaiting a validated real dataset, so no real-data benchmark figures are reported.